# Выполнение ЛР №3: Введение в построение признаков

## Подключение библиотек

In [ ]:
import pandas               as pd
import numpy                as np

import matplotlib           as mpl
import matplotlib.pyplot    as mpl_plt

## Настройка библиотек

In [ ]:
mpl_plt.show()

pd.set_option('display.max_rows', None)

## Задание 1

### Формулировка

* Изучить  датасет и контекст (при наличии) в папке с 
вариантом:  определить  типы  данных  признаков  и 
назначение

### Решение

In [ ]:
from pathlib import Path

DATA_PATH = Path('Вариант 4') / 'OnlineRetail.csv'

print(f'Путь к датасету: "{DATA_PATH.resolve()}"')
print(f'Размер файла: {DATA_PATH.stat().st_size / 1024:.1f} КБ')

retail_df = pd.read_csv(DATA_PATH, encoding='latin1')

print('\nПервые строки:')
display(retail_df.head())

print('\nИнформация о столбцах:')
retail_df.info()

display(retail_df.describe(include=['int64', 'float64']))
display(retail_df.describe(include=['object']))

### Замечание

Обнаружено несоответствие между описанием в [context.md](./Вариант%204/context.md) (геометрия зерен пшеницы) и фактическим датасетом транзакций;

### Расшифровка признаков датасета

#### InvoiceNo (Номер счета-фактуры)

- **Тип признака:** категориальный
- **Назначение:**   номер счета-фактуры, объединяет несколько строк одной покупки.

* * *

#### StockCode (Каталожный артикул)

- **Тип признака:** категориальный
- **Назначение:**   каталожный артикул, используется для группировки и агрегаций по товару.

* * *

#### Quantity (Количество)

- **Тип признака:** числовой
- **Назначение:**   количество единиц товара в строке транзакции, используется для расчета объемов продаж.

* * *

#### InvoiceDate (Дата формирования счета)

- **Тип признака:** числовой 
- **Назначение:**   дата и время оформления счёта, источник временных признаков.

* * *

#### UnitPrice (Цена)

- **Тип признака:** числовой
- **Назначение:**   цена за единицу товара, участвует в расчете выручки.

* * *

#### CustomerID (Идентификатор клиента)

- **Тип признака:** категориальный
- **Назначение:**   уникальный идентификатор клиента, используется для сегментации и RFM-анализа.

* * *

#### Country (Страна)

- **Тип признака:** категориальный
- **Назначение:** страна клиента, применяется в географическом анализе.

## Задание 2

### Формулировка

* Обнаружить  и  обработать выбросы  в  значениях 
признаков. 

### Решение

## Задание 3

### Формулировка

* Обработать  пропущенные  значения, подобрав 
подходящий алгоритм.

### Решение

#### Анализ пропущенных значений

Сначала проанализируем, какие признаки содержат пропущенные значения и в каком количестве.

In [ ]:
# Анализ пропущенных значений
print("Анализ пропущенных значений:")
print("=" * 60)

missing_analysis = pd.DataFrame({
    'Количество пропусков': retail_df.isnull().sum(),
    'Процент пропусков': (retail_df.isnull().sum() / len(retail_df) * 100).round(2),
    'Количество заполненных': retail_df.notna().sum()
})
missing_analysis = missing_analysis[missing_analysis['Количество пропусков'] > 0].sort_values('Количество пропусков', ascending=False)

display(missing_analysis)

print("\n" + "=" * 60)
print("Детальный анализ пропусков по признакам:\n")

# Анализ Description
print("1. Description (Описание товара):")
desc_missing = retail_df['Description'].isnull().sum()
desc_pct = (desc_missing / len(retail_df) * 100)
print(f"   Пропусков: {desc_missing} ({desc_pct:.2f}%)")
print(f"   Заполнено: {retail_df['Description'].notna().sum()} ({100-desc_pct:.2f}%)")

# Проверим, есть ли связь между пропусками Description и другими признаками
print("\n   Анализ связей с другими признаками:")
desc_missing_df = retail_df[retail_df['Description'].isnull()]
print(f"   - Уникальных StockCode с пропущенным Description: {desc_missing_df['StockCode'].nunique()}")
print(f"   - Уникальных InvoiceNo с пропущенным Description: {desc_missing_df['InvoiceNo'].nunique()}")

# Анализ CustomerID
print("\n2. CustomerID (Идентификатор клиента):")
cust_missing = retail_df['CustomerID'].isnull().sum()
cust_pct = (cust_missing / len(retail_df) * 100)
print(f"   Пропусков: {cust_missing} ({cust_pct:.2f}%)")
print(f"   Заполнено: {retail_df['CustomerID'].notna().sum()} ({100-cust_pct:.2f}%)")

# Проверим, есть ли связь между пропусками CustomerID и другими признаками
print("\n   Анализ связей с другими признаками:")
cust_missing_df = retail_df[retail_df['CustomerID'].isnull()]
print(f"   - Уникальных InvoiceNo с пропущенным CustomerID: {cust_missing_df['InvoiceNo'].nunique()}")
print(f"   - Уникальных Country с пропущенным CustomerID: {cust_missing_df['Country'].nunique()}")

# Проверим, пересекаются ли пропуски
print("\n3. Анализ пересечения пропусков:")
both_missing = retail_df[retail_df['Description'].isnull() & retail_df['CustomerID'].isnull()]
print(f"   Строк с пропусками в обоих признаках: {len(both_missing)}")

# Примеры строк с пропусками
print("\n4. Примеры строк с пропущенными значениями:")
mask = retail_df.isnull().any(axis=1)
display(retail_df.loc[mask].head(10))


#### Выбор алгоритма обработки пропущенных значений

**Обоснование выбора стратегии:**

1. **Description (Описание товара)** - ~0.27% пропусков:
   - **Стратегия:** Заполнение по StockCode (каталожному артикулу)
   - **Обоснование:** 
     - Описание товара тесно связано с его артикулом (StockCode)
     - Один и тот же артикул должен иметь одинаковое описание
     - Можно использовать наиболее частое описание для данного StockCode
     - Если для StockCode нет описания, используем сам StockCode как описание
   - **Алгоритм:** `fillna()` с группировкой по StockCode и применением `mode()` или `first()`

2. **CustomerID (Идентификатор клиента)** - ~24.9% пропусков:
   - **Стратегия:** Создание специальной категории для анонимных клиентов
   - **Обоснование:**
     - Большой процент пропусков (почти 25%) указывает на систематический характер (возможно, анонимные покупки)
     - CustomerID - категориальный идентификатор, его нельзя интерполировать
     - Заполнение средним/медианой не имеет смысла для идентификатора
     - Создание отдельной категории (например, 0 или -1) позволит сохранить информацию о наличии пропуска
   - **Алгоритм:** `fillna()` с специальным значением (0 или -1) с последующим преобразованием в категориальный тип


In [ ]:
# Создаем копию датасета для обработки
retail_df_processed = retail_df.copy()

print("Обработка пропущенных значений:")
print("=" * 60)

# 1. Обработка Description
print("\n1. Обработка Description (Описание товара):")
print(f"   Пропусков до обработки: {retail_df_processed['Description'].isnull().sum()}")

# Заполняем пропуски Description на основе StockCode
# Для каждого StockCode находим наиболее частое описание
stockcode_to_description = retail_df_processed.groupby('StockCode')['Description'].apply(
    lambda x: x.mode().iloc[0] if not x.mode().empty else None
).to_dict()

# Заполняем пропуски
def fill_description(row):
    if pd.isna(row['Description']):
        stock_code = row['StockCode']
        if stock_code in stockcode_to_description and stockcode_to_description[stock_code] is not None:
            return stockcode_to_description[stock_code]
        else:
            # Если для StockCode нет описания, используем сам StockCode
            return f"Unknown Item ({stock_code})"
    return row['Description']

retail_df_processed['Description'] = retail_df_processed.apply(fill_description, axis=1)

print(f"   Пропусков после обработки: {retail_df_processed['Description'].isnull().sum()}")

# 2. Обработка CustomerID
print("\n2. Обработка CustomerID (Идентификатор клиента):")
print(f"   Пропусков до обработки: {retail_df_processed['CustomerID'].isnull().sum()}")

# Заполняем пропуски специальным значением для анонимных клиентов
# Используем 0 как идентификатор анонимного клиента
retail_df_processed['CustomerID'] = retail_df_processed['CustomerID'].fillna(0)

# Преобразуем в целочисленный тип (так как это идентификатор)
retail_df_processed['CustomerID'] = retail_df_processed['CustomerID'].astype(int)

print(f"   Пропусков после обработки: {retail_df_processed['CustomerID'].isnull().sum()}")
print(f"   Уникальных CustomerID (включая анонимных): {retail_df_processed['CustomerID'].nunique()}")
print(f"   Количество анонимных клиентов (CustomerID=0): {(retail_df_processed['CustomerID'] == 0).sum()}")

# Проверка результата
print("\n" + "=" * 60)
print("Итоговая проверка пропущенных значений:")
print("=" * 60)
final_missing = retail_df_processed.isnull().sum()
final_missing = final_missing[final_missing > 0]
if len(final_missing) == 0:
    print("✓ Все пропущенные значения успешно обработаны!")
else:
    print("Остались пропуски в следующих признаках:")
    display(final_missing)

print("\nИнформация о датасете после обработки:")
retail_df_processed.info()


#### Визуализация результатов обработки

Проверим качество обработки и покажем примеры заполненных значений.


In [ ]:
# Визуализация результатов обработки

# 1. Сравнение до и после обработки
comparison = pd.DataFrame({
    'До обработки': [retail_df['Description'].isnull().sum(), retail_df['CustomerID'].isnull().sum()],
    'После обработки': [retail_df_processed['Description'].isnull().sum(), retail_df_processed['CustomerID'].isnull().sum()]
}, index=['Description', 'CustomerID'])

print("Сравнение количества пропусков до и после обработки:")
display(comparison)

# 2. Примеры заполненных значений Description
print("\nПримеры заполненных значений Description:")
# Найдем строки, где Description был заполнен
original_missing_desc = retail_df['Description'].isnull()
filled_desc = retail_df_processed[original_missing_desc][['StockCode', 'Description']].head(10)
display(filled_desc)

# 3. Статистика по CustomerID
print("\nСтатистика по CustomerID:")
cust_stats = pd.DataFrame({
    'Категория': ['Идентифицированные клиенты', 'Анонимные клиенты (CustomerID=0)'],
    'Количество транзакций': [
        (retail_df_processed['CustomerID'] != 0).sum(),
        (retail_df_processed['CustomerID'] == 0).sum()
    ],
    'Процент': [
        (retail_df_processed['CustomerID'] != 0).sum() / len(retail_df_processed) * 100,
        (retail_df_processed['CustomerID'] == 0).sum() / len(retail_df_processed) * 100
    ]
})
display(cust_stats)

# 4. Проверка качества заполнения Description
print("\nПроверка качества заполнения Description:")
# Проверим, сколько уникальных StockCode получили описания
stockcodes_with_desc_before = retail_df[retail_df['Description'].notna()]['StockCode'].nunique()
stockcodes_with_desc_after = retail_df_processed['StockCode'].nunique()
print(f"Уникальных StockCode с описанием до обработки: {stockcodes_with_desc_before}")
print(f"Уникальных StockCode с описанием после обработки: {stockcodes_with_desc_after}")

# Примеры первых строк обработанного датасета
print("\nПервые строки обработанного датасета:")
display(retail_df_processed.head(10))


## Задание 4

### Формулировка

* Выполнить  преобразование  категориальных  данных 
(при наличии), подобрав подходящие алгоритмы.

### Решение

## Задание 5

### Формулировка

* К временным данным следует применить квантование.

### Решение